# 01. Khảo sát dữ liệu (EDA) - Dự báo nhu cầu điện năng

**Mục tiêu:** dự báo lượng điện (kW trung bình) của **giờ kế tiếp** từ **24 giờ gần nhất** cho một khách hàng.

Notebook này trả lời bốn câu hỏi: dữ liệu có sạch không, **chọn chuỗi nào và dựa trên tiêu chí gì**, chia tập ra sao, và có điểm bất thường nào cần ghi nhớ cho phần phân tích lỗi.

**Cách chạy:** mở notebook trong thư mục `notebooks/`, chạy lần lượt từ trên xuống. Cần có trước:
`data/processed/series_hourly.csv` (tạo bởi `python src/data_preprocessing.py`) và các bảng chọn chuỗi
trong `results/tables/` (tạo bởi `python src/select_customers.py`).

**Quy ước dữ liệu:** giá trị là kW trung bình mỗi giờ; nhãn thời gian là *mốc kết thúc* của giờ
(mốc 02:00 gồm 01:15, 01:30, 01:45 và 02:00).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import load_config, path_from, set_seed
from src.data import load_hourly_data, split_by_time, split_summary
from src.dst_check import dst_table

cfg = load_config()
set_seed(cfg["project"]["seed"])

FIG_DIR = path_from(cfg, "figures_dir"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = path_from(cfg, "tables_dir");  TAB_DIR.mkdir(parents=True, exist_ok=True)
MAIN = cfg["customers"]["main"]

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
print("Chuỗi chính:", MAIN, "| Các chuỗi dùng:", cfg["customers"]["hourly_file_columns"])

## 1. Tổng quan dữ liệu theo giờ

In [ ]:
df = load_hourly_data(cfg)   # kiểm tra: không trùng, không thiếu, không âm, cách đều 1 giờ
train, val, test = split_by_time(df, cfg)

print("Số giờ:", len(df), "| từ", df.index.min(), "đến", df.index.max())
print("Giá trị thiếu:", int(df.isna().sum().sum()), "| giá trị bằng 0:", int((df == 0).sum().sum()))
df.describe().round(2)

Dữ liệu có đúng 35.064 giờ (4 năm 2011-2014), không thiếu, không trùng, không giá trị 0 và không âm ở cả 4 chuỗi.

Việc kiểm tra này được tự động hóa trong `load_hourly_data`: nếu dữ liệu bị lỗi, hàm **báo lỗi và dừng**
thay vì chạy tiếp với dữ liệu sai. Bốn chuỗi trải trên dải tải rộng, từ khoảng 40 kW (MT_169) đến
khoảng 641 kW (MT_294) - chênh nhau khoảng 16 lần.

## 2. Phân phối giá trị của chuỗi chính

In [ ]:
s = df[MAIN]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(s.values, bins=60, color="tab:blue", alpha=0.85)
axes[0].set(title=f"{MAIN}: phân phối giá trị theo giờ", xlabel="kW", ylabel="Số giờ")

by_dow = [s[s.index.dayofweek == d].values for d in range(7)]
axes[1].boxplot(by_dow, tick_labels=["T2", "T3", "T4", "T5", "T6", "T7", "CN"], showfliers=False)
axes[1].set(title=f"{MAIN}: phân phối theo thứ trong tuần", ylabel="kW")

fig.tight_layout(); fig.savefig(FIG_DIR / "eda_main_distribution.png", dpi=150); plt.show()

print(s.describe().round(2).to_string())
print("\nTỷ lệ giờ dưới 40 kW:", f"{(s < 40).mean():.1%}")

**Nhận xét**

Phân phối **hai đỉnh (bimodal)** chứ không phải hình chuông: một cụm dày ở mức thấp, tập trung quanh 22-28 kW,
và một cụm thứ hai ở mức cao quanh 96-102 kW. Cụ thể, 38,5% số giờ nằm dưới 40 kW và 38,0% nằm trên 90 kW,
còn khoảng giữa 40-90 kW chỉ chiếm 23,4%. Đây không phải nhiễu mà là hai chế độ vận hành khác nhau của
khách hàng - thấy rõ ở biểu đồ hộp bên phải: thứ Hai đến thứ Sáu nằm ở mức cao, thứ Bảy thấp hơn hẳn và
Chủ nhật thấp nhất.

Hệ quả cho mô hình: **sai số sẽ không phân bố đều**. Những giờ chuyển giữa hai chế độ (sáng thứ Hai, chiều thứ Sáu)
là nơi mô hình dễ sai nhất, và đó là chỗ cần soi khi phân tích lỗi. Phân phối lệch như vậy cũng là lý do
nhóm cân nhắc hàm mất mát Huber thay cho MSE nếu MSE bị các giá trị đỉnh chi phối quá mức.

## 3. Quy trình chọn chuỗi: 370 chuỗi còn 4

In [ ]:
# Các bảng do src/select_customers.py tạo ra. Tiêu chí và ngưỡng nằm ở mục `selection`
# trong configs/config.yaml, không đặt cứng trong code.
for name in ["customer_selection", "customer_main_candidates"]:
    p = TAB_DIR / f"{name}.csv"
    if p.exists():
        print(f"--- {name}.csv ---")
        display(pd.read_csv(p, encoding="utf-8-sig").set_index("customer").round(3))
    else:
        print(f"Chưa có {p.name}. Chạy: python src/select_customers.py")

In [ ]:
# Tính lại đặc trưng cho 4 chuỗi đang dùng, để notebook tự kiểm chứng bảng ở trên.
def longest_constant_run(s):
    same = s.diff() == 0
    if not same.any():
        return 0, 0
    runs = same.groupby((~same).cumsum()).sum()
    return int(runs.max()), int((runs >= 6).sum())

rows = []
for c in df.columns:
    x = df[c]
    run_max, run6 = longest_constant_run(x)
    rows.append({"customer": c, "mean_kW": x.mean(), "cv(std/mean)": x.std() / x.mean(),
                 "max_kW": x.max(), "giờ_không_đổi_dài_nhất": run_max, "số_đoạn_không_đổi_>=6h": run6,
                 "autocorr_lag1": x.autocorr(1), "autocorr_lag24": x.autocorr(24),
                 "autocorr_lag168": x.autocorr(168)})

compare = pd.DataFrame(rows).set_index("customer").round(3)
compare.to_csv(TAB_DIR / "customer_comparison_hourly.csv", encoding="utf-8-sig")
compare

**Quy trình chọn chuỗi**

Bộ dữ liệu có 370 khách hàng. Nhóm lọc qua bốn tầng, mọi ngưỡng đặt trong `configs/config.yaml`
(mục `selection`) nên chạy lại cho cùng kết quả:

| Tầng | Lọc theo | Còn lại |
|---|---|---|
| A | chất lượng dữ liệu thô: thiếu, âm, bắt đầu muộn, giá trị 0 giữa chuỗi | 370 → 90 |
| C | đặc trưng theo giờ: đoạn phẳng, cv, chu kỳ tuần, dịch mức tải | 90 → 78 |
| D1 | **chuỗi chính**: `ac_lag168` cao nhất trong nhóm `ac_lag24 < 0,90` | 78 → 1 |
| D2 | chuỗi phụ trải đều theo mức tải | 78 → 3 |

Ở tầng A, 197 chuỗi bị loại vì vừa bắt đầu muộn vừa có giá trị 0 giữa chuỗi, 68 chuỗi chỉ vì giá trị 0
giữa chuỗi, 15 chuỗi chỉ vì bắt đầu muộn. Giá trị 0 ở **đầu** chuỗi là giai đoạn khách hàng chưa có dữ liệu
(theo mô tả của bộ dữ liệu), khác hẳn ý nghĩa với giá trị 0 nằm giữa chuỗi, nên nhóm tách thành hai tiêu chí riêng.

**Vì sao chọn MT_158 làm chuỗi chính**

Tiêu chí ở tầng D1 được đặt **trước khi chạy bất kỳ mô hình nào**: chuỗi chính phải là chuỗi mà baseline
Seasonal Naive 24 giờ *chưa* gần như tối ưu. Lý do: nếu `ac_lag24` quá cao thì "giá trị cùng giờ hôm qua"
đã bám sát chuỗi, mô hình học sâu gần như không còn dư địa cải thiện và bài toán mất ý nghĩa thực nghiệm.

Kết quả: **chỉ 2 trên 78 chuỗi** thỏa điều kiện `ac_lag24 < 0,90`. Phần lớn khách hàng trong bộ dữ liệu có
chu kỳ ngày lặp gần như máy móc. MT_158 đứng đầu nhóm đó với `ac_lag24 = 0,560` và `ac_lag168 = 0,891`,
tức là có chu kỳ **tuần** rất rõ nhưng chu kỳ **ngày** không lặp máy móc - đúng loại chuỗi cần cho bài toán.
MT_180 (`ac_lag24 = 0,865`) là phương án dự phòng.

**Vai trò của từng chuỗi**

| Chuỗi | Mức tải | `ac_lag24` | Vai trò |
|---|---|---|---|
| **MT_158** | 76 kW | 0,560 | **chuỗi chính** |
| MT_169 | 40 kW | 0,950 | kiểm tra tổng quát ở quy mô nhỏ |
| MT_168 | 137 kW | 0,959 | kiểm tra tổng quát cùng quy mô |
| MT_294 | 641 kW | 0,944 | kiểm tra tổng quát ở quy mô gấp ~8,5 lần |

Nhóm dùng 4 chuỗi, nằm trong phạm vi 1-5 chuỗi đề bài cho phép. Tầng D2 đề xuất thêm MT_262 (197 kW)
nhưng chuỗi này trùng vai trò "khách hàng tải lớn" với MT_294 nên không bổ sung thông tin mới.

Hai chuỗi bị loại khỏi danh sách ban đầu: **MT_156** có đoạn giá trị không đổi dài tới 215 giờ liên tiếp
(ít thông tin để học), **MT_159** có `year_drift = 0,83` và lệch mức tải test/train = 0,30 (phân phối
dịch chuyển quá mạnh giữa các tập).

## 4. Chuỗi chính: hình dạng theo thời gian

In [ ]:
train_end, val_end = pd.Timestamp(cfg["split"]["train_end"]), pd.Timestamp(cfg["split"]["val_end"])

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(s.index, s.values, lw=0.5)
ax.axvline(train_end, color="tab:red", ls="--", label="cuối train")
ax.axvline(val_end, color="tab:green", ls="--", label="cuối validation")
ax.set(title=f"{MAIN}: toàn kỳ theo giờ", xlabel="Thời gian", ylabel="kW trung bình mỗi giờ")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_main_full_series.png", dpi=150); plt.show()

Chuỗi dao động trong biên độ lớn với chu kỳ đều, không có khoảng trống, không có đoạn phẳng dài.
Mức nền ban đêm thay đổi nhẹ theo năm (xem mục 7). Hai đường đứt là mốc cắt train/validation/test.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

week = s.loc["2013-05-06":"2013-05-12"]
axes[0].plot(week.index, week.values)
axes[0].set(title=f"{MAIN}: một tuần mẫu (6-12/05/2013)", ylabel="kW")
axes[0].tick_params(axis="x", rotation=30)

wd = s.groupby(s.index.dayofweek).mean()
axes[1].bar(["T2", "T3", "T4", "T5", "T6", "T7", "CN"], wd.values)
axes[1].set(title=f"{MAIN}: trung bình theo thứ trong tuần", ylabel="kW")

fig.tight_layout(); fig.savefig(FIG_DIR / "eda_main_week_and_weekday.png", dpi=150); plt.show()
wd.round(1).rename(index=dict(enumerate(["T2", "T3", "T4", "T5", "T6", "T7", "CN"])))

Ngày thường (thứ Ba đến thứ Sáu) có mức tiêu thụ trung bình cao, khoảng 95 kW; thứ Bảy thấp hơn nhiều,
khoảng 44 kW; Chủ nhật thấp nhất, khoảng 24 kW.

Đây là lý do **Seasonal Naive lag 168** (cùng giờ tuần trước) hợp lý hơn lag 24 cho chuỗi này: lấy giá trị
của hôm qua sẽ sai nặng ở các mốc chuyển giữa ngày thường và cuối tuần. Cũng vì vậy mà cửa sổ đầu vào
168 giờ đáng được thử ở bước thí nghiệm cải tiến.

In [ ]:
h = s.index.hour
is_weekend = s.index.dayofweek >= 5

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(24), s[~is_weekend].groupby(h[~is_weekend]).mean(), marker="o", label="ngày thường")
ax.plot(range(24), s[is_weekend].groupby(h[is_weekend]).mean(), marker="o", label="cuối tuần")
ax.set(title=f"{MAIN}: trung bình theo nhãn giờ trong ngày", xlabel="Nhãn giờ (mốc kết thúc)", ylabel="kW")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_main_hour_profile.png", dpi=150); plt.show()

Ngày thường: mức cao vào buổi tối và đầu giờ sáng (nhãn khoảng 19-02), thấp nhất khoảng nhãn 03-08,
tăng nhảy bậc ở nhãn 09 rồi giữ mức trung bình ban ngày. Có một điểm tụt riêng ở nhãn 22 rồi bật lại ở nhãn 23;
chưa rõ nguyên nhân, có thể do lịch vận hành hoặc biểu giá điện của khách hàng - đây **chỉ là giả thuyết**,
cần kiểm chứng thêm trước khi kết luận. Cuối tuần thấp và phẳng hơn nhiều.

Lưu ý khi đọc hình: nhãn là *mốc kết thúc* của giờ, nên "nhãn 09" ứng với khoảng 08:15-09:00.

## 5. Tự tương quan: căn cứ chọn độ dài cửa sổ

In [ ]:
max_lag = 192   # 8 ngày, đủ thấy cả chu kỳ ngày (24) và chu kỳ tuần (168)
lags = range(1, max_lag + 1)
acf = [s.autocorr(k) for k in lags]

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(list(lags), acf, width=1.0)
for k, c in [(24, "tab:red"), (48, "tab:orange"), (168, "tab:green")]:
    ax.axvline(k, color=c, ls="--", lw=1.2, label=f"lag {k}")
ax.axhline(0, color="k", lw=0.8)
ax.set(title=f"{MAIN}: tự tương quan theo độ trễ (giờ)", xlabel="Độ trễ (giờ)", ylabel="Hệ số tự tương quan")
ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_main_acf.png", dpi=150); plt.show()

print("Các độ trễ đáng chú ý:")
for k in [1, 2, 3, 24, 48, 72, 120, 144, 168]:
    print(f"  lag {k:>3}: {s.autocorr(k):.3f}")

**Nhận xét**

Hình này là căn cứ trực tiếp cho ba quyết định kỹ thuật của nhóm:

1. **Tự tương quan lag 1 rất cao (0,878).** Giá trị giờ liền trước giải thích được phần lớn giá trị giờ kế tiếp.
   Đây là lý do baseline **Last Value** sẽ rất mạnh, và mô hình học sâu phải vượt được nó mới có ý nghĩa.
2. **Đỉnh ở lag 168 (0,891) cao hơn lag 24 (0,560).** Chu kỳ tuần mạnh hơn chu kỳ ngày - khớp với biểu đồ
   theo thứ ở mục 4. Vì vậy Seasonal Naive 168 được kỳ vọng tốt hơn Seasonal Naive 24.
3. **Hệ số tụt sâu ở các độ trễ giữa tuần rồi bật lên cuối tuần.** Từ 0,560 ở lag 24, hệ số rơi xuống khoảng
   0,21 ở các lag 48-120, rồi bật lên 0,542 ở lag 144 và đạt đỉnh 0,891 ở lag 168. Cửa sổ 24 giờ nắm được chu kỳ ngày,
   nhưng phải tới cửa sổ 168 giờ mô hình mới quan sát trọn một chu kỳ tuần. Đó là lý do thí nghiệm cải tiến
   so sánh ba độ dài cửa sổ 24, 48 và 168.

Khi so sánh các độ dài cửa sổ, phải đánh giá trên **cùng một tập mốc thời gian**, nếu không số mẫu khác nhau
sẽ làm kết quả không so sánh được.

## 6. So sánh hình dạng bốn chuỗi

In [ ]:
# Chuẩn hóa z-score theo từng chuỗi để so sánh HÌNH DẠNG, không so sánh độ lớn.
week_all = df.loc["2013-05-06":"2013-05-12"]
norm = (week_all - week_all.mean()) / week_all.std()

fig, ax = plt.subplots(figsize=(14, 4.5))
for c in norm.columns:
    ax.plot(norm.index, norm[c], lw=1.4 if c == MAIN else 0.9,
            alpha=1.0 if c == MAIN else 0.6,
            label=f"{c} (chính)" if c == MAIN else c)
ax.set(title="Một tuần mẫu (6-12/05/2013), đã chuẩn hóa z-score theo từng chuỗi",
       xlabel="Thời gian", ylabel="Độ lệch chuẩn so với trung bình tuần")
ax.tick_params(axis="x", rotation=20); ax.legend(ncol=4)
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_series_comparison.png", dpi=150); plt.show()

**Nhận xét**

Chuẩn hóa z-score cho từng chuỗi để so sánh *hình dạng* thay vì độ lớn. Khác biệt lộ ra rõ:

- **MT_158** (đường đậm) có biên độ ngày-đêm lớn và sụt mạnh vào cuối tuần, các đỉnh không lặp lại giống nhau
  giữa các ngày. Đây chính là điều mà `ac_lag24 = 0,560` phản ánh bằng số.
- **MT_168, MT_169, MT_294** có dạng gần như lặp lại từng ngày, biên độ đều đặn. Với các chuỗi này,
  Seasonal Naive 24 giờ gần như đã là lời giải, nên chúng chỉ đóng vai trò kiểm tra tính tổng quát
  của pipeline chứ không phải nơi so tài giữa các mô hình.

Đây là bằng chứng bằng hình cho quyết định chọn chuỗi chính ở mục 3.

## 7. Mức tải theo năm và khác biệt giữa train / validation / test

In [ ]:
# Nhãn là mốc kết thúc, nên giờ 00:00 ngày 01/01/2015 thuộc về năm 2014.
year = (df.index - pd.Timedelta(hours=1)).year
by_year = df.groupby(year).mean().round(1)
by_year.index.name = "năm"
display(by_year)

by_split = pd.DataFrame({"train": train.mean(), "validation": val.mean(), "test": test.mean()}).T.round(1)
display(by_split)

fig, ax = plt.subplots(figsize=(14, 4))
for c in df.columns:
    ax.plot(df[c].rolling(24 * 30).mean(), label=c)
ax.axvline(train_end, color="k", ls="--", lw=0.8)
ax.axvline(val_end, color="k", ls="--", lw=0.8)
ax.set(title="Trung bình trượt 30 ngày của các chuỗi", ylabel="kW")
ax.legend(ncol=4)
fig.tight_layout(); fig.savefig(FIG_DIR / "eda_rolling_mean_all.png", dpi=150); plt.show()

**Nhận xét**

- **MT_158**: trung bình theo năm khoảng 89,8 (2011), 67-68 (2012-2013) và 76,7 (2014). Chỉ số `year_drift`
  bằng 0,296 - cao nhất trong bốn chuỗi đang dùng. Tập validation và test rơi vào phần có mức nền khác so với
  phần lớn tập train.
- **MT_168, MT_169, MT_294**: ổn định hơn nhiều (`year_drift` lần lượt 0,076; 0,057; 0,182), phù hợp với vai trò
  kiểm tra tính tổng quát.

**Cần ghi vào phần hạn chế của báo cáo:** dữ liệu có dịch mức theo năm; validation (tháng 3-7/2014) và
test (tháng 8-12/2014) thuộc các mùa khác nhau và khác mức tải so với phần lớn tập train. Đây là hệ quả
tất yếu của việc chia theo thời gian - đổi lại, cách chia này mô phỏng đúng tình huống thực tế là dùng
quá khứ để dự báo tương lai, và loại bỏ rò rỉ dữ liệu.

## 8. Chia train / validation / test (80/10/10 theo thời gian)

In [ ]:
split_tbl = split_summary({"train": train, "validation": val, "test": test})
display(split_tbl.iloc[:, :5])
print("Tổng:", len(train) + len(val) + len(test), "giờ")

Mốc cắt làm tròn theo ranh giới tháng, cắt theo thời gian và **không xáo trộn**: train 27.720 giờ (79,1%),
validation 3.672 giờ (10,5%), test 3.672 giờ (10,5%); validation và test mỗi tập 153 ngày.

Hai quy ước quan trọng cho các bước sau:

1. Khi tạo cửa sổ trượt, 24 giờ đầu vào của những mẫu đầu tập validation/test được lấy từ **cuối tập liền trước**
   (dữ liệu quá khứ đã quan sát, hoàn toàn hợp lệ), nhưng **nhãn dự báo phải nằm đúng trong tập của nó**.
   Nhờ vậy mọi độ dài cửa sổ đều cho ra đúng 3.672 nhãn và so sánh được công bằng.
2. Bộ chuẩn hóa chỉ được fit trên tập train, và **mỗi chuỗi dùng một scaler riêng** để còn
   `inverse_transform` đúng về thang đo gốc.

## 9. Giờ bất thường quanh ngày chuyển giờ (DST)

In [ ]:
dst = dst_table(df, cfg)
dst.to_csv(TAB_DIR / "dst_hours.csv", index=False)

display(dst[dst.customer == MAIN].drop(columns="customer"))
print("Số ngày được gợi ý kiểm tra, theo chuỗi:")
print(dst.groupby("customer")["suggest_check"].sum().to_string())

**Nhận xét**

- Vào các ngày chuyển giờ **tháng 3** (2012, 2013, 2014), giờ nhãn 02:00 tụt xuống chỉ còn khoảng 0,13-0,35 lần
  so với các giờ lân cận; vào ngày chuyển giờ **tháng 10**, giờ này tăng khoảng 1,4-1,6 lần vì hai giờ bị gộp làm một.
  Hiện tượng khớp với mô tả của bộ dữ liệu.
- Cả bốn chuỗi đều gợi ý đúng **6 ngày**, và sáu ngày đó trùng khít với các ngày chuyển giờ 2012-2014.
  Bốn chuỗi độc lập cùng chỉ vào một tập ngày là bằng chứng đủ mạnh để kết luận.
- Riêng MT_158, hai mốc năm 2011 không vượt ngưỡng (tỷ lệ 0,75 và 0,90). Với chuỗi biến động mạnh như MT_158,
  cột `suggest_check` chỉ là *gợi ý*, không phải kết luận.
- Mốc tháng 3/2014 nằm trong tập **validation**, mốc tháng 10/2014 nằm trong tập **test**. Chúng có thể tạo ra
  lỗi dự báo lớn.

**Quyết định: giữ nguyên dữ liệu**, không sửa và không loại bỏ. Các mốc này được ghi lại trong
`results/tables/dst_hours.csv` và dùng làm **giả thuyết** giải thích lỗi ở bước phân tích lỗi -
chỉ kết luận khi có bằng chứng từ chính dữ liệu dự báo.

## 10. Kết luận và việc tiếp theo

**Quyết định về dữ liệu**

1. **Chuỗi chính: MT_158**, chọn theo quy trình bốn tầng với ngưỡng ghi trong `configs/config.yaml`.
   Là 1 trong 2 chuỗi (trên 78 chuỗi đạt chuẩn) có `ac_lag24 < 0,90`. Dự phòng: MT_180.
2. **Chuỗi phụ: MT_169 (40 kW), MT_168 (137 kW), MT_294 (641 kW)** - kiểm tra tính tổng quát của pipeline
   ở ba quy mô tải khác nhau.
3. Dữ liệu theo giờ (kW trung bình, nhãn là mốc kết thúc), 35.064 giờ, không thiếu, không giá trị 0, không âm.
4. Chia theo thời gian: train đến 2014-03-01 00:00, validation đến 2014-08-01 00:00, phần còn lại là test
   (27.720 / 3.672 / 3.672 giờ).
5. Baseline cần có: Last Value, Moving Average, Seasonal Naive lag 24 và lag 168.

**Hạn chế cần nêu trong báo cáo**

- Mức tải dịch chuyển theo năm, rõ nhất ở chính chuỗi MT_158 (`year_drift = 0,296`).
- Validation và test rơi vào các mùa khác nhau và khác mức nền so với phần lớn tập train.
- Các giờ bất thường quanh ngày chuyển giờ nằm trong cả validation và test.
- Tự tương quan lag 1 rất cao (0,878) nên baseline Last Value là đối thủ mạnh; nếu mô hình học sâu
  không vượt được, nhóm **báo cáo trung thực và phân tích nguyên nhân**, không điều chỉnh theo tập test.

**Việc tiếp theo**

Baseline và bộ hàm đánh giá (`src/baseline.py`, `src/metrics.py`, `notebooks/02_baseline.ipynb`),
tạo cửa sổ trượt dùng chung, rồi đến MLP và LSTM.